# 06 — The Transformer Block
### Starter Notebook

**Learning objectives**
- Combine attention + FFN into a full transformer block
- Understand Pre-LN vs Post-LN and why modern LLMs use Pre-LN
- Implement RMSNorm
- Stack blocks into a complete decoder-only model and run a forward pass
- Count and analyse model parameters

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — The Anatomy of a Transformer Block

A single transformer block applies (in Pre-LN order):

```
x → LayerNorm → CausalSelfAttention → + (residual) →
  → LayerNorm → FeedForward         → + (residual) → output
```

The **residual connections** are crucial — they give gradients a direct path back to early layers.

### Exercise A1 — Implement RMSNorm

In [ ]:
class MyRMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalisation.

    Simpler than LayerNorm — no mean subtraction, only RMS scaling.
    Used in LLaMA, Gemma, PaLM.

    RMSNorm(x) = x / sqrt(mean(x²) + ε) * γ
    """

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # TODO: learnable scale parameter γ, initialised to ones

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: compute RMS and normalise
        pass


norm = MyRMSNorm(64)
x_test = torch.randn(2, 10, 64)
out_test = norm(x_test)
if out_test is not None:
    print(f'RMSNorm output shape: {out_test.shape}')
    # Check that variance of output is ~1 before the scale
    rms_of_output = out_test.pow(2).mean(-1).sqrt()
    print(f'Mean RMS of output: {rms_of_output.mean():.4f}   # should be ~1.0')
else:
    print('Not implemented yet.')

## Part B — Pre-LN vs Post-LN

- **Post-LN** (original paper): normalise *after* the residual add. Training is unstable without careful warmup.
- **Pre-LN** (modern default): normalise *before* attention/FFN. Much more stable training.

### Exercise B1 — Implement a Pre-LN Decoder Block

In [ ]:
from src.models.attention import CausalSelfAttention
from src.models.feedforward import FeedForward


class MyDecoderBlock(nn.Module):
    """
    Pre-LN decoder-only transformer block.

    Forward:
        x = x + CausalSelfAttention(RMSNorm(x))
        x = x + FeedForward(RMSNorm(x))
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int = None, dropout: float = 0.0):
        super().__init__()
        # TODO: create norm1, norm2, attention, ffn

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement Pre-LN forward pass with residual connections
        pass


block = MyDecoderBlock(d_model=64, n_heads=4, d_ff=256)
x = torch.randn(2, 16, 64)
out = block(x)
if out is not None:
    print(f'Block output shape: {out.shape}   # expect (2, 16, 64)')
else:
    print('Not implemented yet.')

## Part C — Residual Connection Importance

Demonstrate that removing residual connections collapses gradient flow.

In [ ]:
class BlockNoResidual(nn.Module):
    """Block WITHOUT residual connections — for ablation comparison."""
    def __init__(self, d_model, n_heads, d_ff=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ffn  = FeedForward(d_model, d_ff)

    def forward(self, x):
        # NOTE: no + x residual — output replaces input entirely
        x, _ = self.attn(self.norm1(x))
        x = self.ffn(self.norm2(x))
        return x


def stack_and_measure_grad(block_cls, n_layers=8, d_model=64, **kwargs):
    """Stack n_layers blocks, run forward+backward, return embedding grad norm."""
    layers = nn.Sequential(*[block_cls(d_model, **kwargs) for _ in range(n_layers)])
    embed = nn.Embedding(100, d_model)
    x = torch.randint(0, 100, (1, 16))
    out = layers(embed(x))
    loss = out.sum()
    loss.backward()
    return embed.weight.grad.norm().item()


# TODO: compare grad norms for blocks with vs without residuals
# Uncomment after implementing MyDecoderBlock:
# with_res = stack_and_measure_grad(MyDecoderBlock,  n_layers=8, n_heads=4, d_ff=256)
# no_res   = stack_and_measure_grad(BlockNoResidual, n_layers=8, n_heads=4)
# print(f'With residuals  : grad norm = {with_res:.4f}')
# print(f'Without residuals: grad norm = {no_res:.6f}')
print('[Exercise C] Uncomment the comparison above after implementing MyDecoderBlock.')

## Part D — Full Model Forward Pass

In [ ]:
from src.models.transformer import TransformerLM, TransformerConfig

cfg = TransformerConfig(
    vocab_size=512,
    d_model=128,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    max_seq_len=128,
    dropout=0.0,
    ffn_type='swiglu',
    norm_type='rmsnorm',
    pos_encoding='rope',
)

model = TransformerLM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

tokens = torch.randint(0, cfg.vocab_size, (2, 32), device=device)
with torch.no_grad():
    logits = model(tokens)
print(f'Input  : {tokens.shape}')
print(f'Output : {logits.shape}   # (batch, seq, vocab)')

## Part E — Parameter Budget Analysis

Where do all the parameters live?

In [ ]:
# Breakdown by component
components = {}
for name, param in model.named_parameters():
    top = name.split('.')[0]
    components[top] = components.get(top, 0) + param.numel()

total = sum(components.values())
print(f'Total: {total:,} parameters\n')
for comp, count in sorted(components.items(), key=lambda x: -x[1]):
    print(f'  {comp:<20} {count:>8,}  ({100*count/total:.1f}%)')

# Pie chart
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    list(components.values()),
    labels=list(components.keys()),
    autopct='%1.1f%%',
    startangle=90
)
ax.set_title('Parameter distribution by component')
plt.tight_layout(); plt.show()

## Summary

The transformer block in two equations:
```
x = x + Attention(Norm(x))   # information mixing across positions
x = x + FFN(Norm(x))         # per-position non-linear transformation
```

**Design choices in modern LLMs:**
- **Pre-LN** (not Post-LN) — better training stability
- **RMSNorm** (not LayerNorm) — simpler, slightly faster
- **SwiGLU** (not GELU FFN) — empirically stronger
- **RoPE** (not sinusoidal) — better length generalisation

**Next:** `../part3_innovations/07_efficient_attention_starter.ipynb`